# Capstone Project BisaKerja


## Menentukan Pertanyaan Bisnis


Dataset yang digunakan dalam analisis ini berasal dari dua sumber utama:
1. **IndoTech-Job Dataset** — Data lowongan kerja aktif di Indonesia hasil scraping dari platform Kalibrr, Kitalulus, Dealls, Glints (Mei 2026) sebanyak 2.099 baris.
2. **TechTalent-Profile Dataset** — Data profil kandidat teknologi (100.000 baris) yang mencakup skill, pengalaman, pendidikan, proyek, dan job role.

Analisis ini bertujuan membangun fondasi data untuk fitur **Job Fit Scoring** dan **Skill Gap Analysis** pada platform BisaKerja.


- **Pertanyaan 1:** Apa 10 skill teknis yang paling banyak diminta oleh lowongan kerja tech di Indonesia berdasarkan data Mei 2026, dan berapa persentase kandidat dalam dataset resume yang sudah memiliki skill tersebut?
- **Pertanyaan 2:** Bagaimana distribusi tingkat pengalaman (Entry/Junior/Mid/Senior) pada lowongan tech di Indonesia, dan apakah kandidat Fresher & 1-2 tahun pengalaman dalam dataset resume sudah memenuhi kualifikasi skill minimum untuk posisi Entry Level?
- **Pertanyaan 3:** Job role tech apa yang paling banyak tersedia di pasar kerja Indonesia saat ini, dan seberapa besar proporsi kandidat dalam dataset resume yang memiliki profil sesuai dengan role tersebut?
- **Pertanyaan 4:** Berapa rata-rata persentase kemiripan (skill overlap) antara profil kandidat dalam dataset resume dengan persyaratan skill lowongan entry-level tech di Indonesia?


## Import Semua Packages/Library yang Digunakan


In [64]:
# ── Data Manipulation ──
import pandas as pd
import numpy as np
import re
import os
from collections import Counter
from pathlib import Path

# ── Visualization ──
import matplotlib.pyplot as plt
import seaborn as sns

# ── HTML Parsing ──
from bs4 import BeautifulSoup

# ── Warning Suppression ──
import warnings
warnings.filterwarnings('ignore')

# ── Display Settings ──
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Semua library berhasil diimport!')


Semua library berhasil diimport!


## Data Wrangling


### Gathering Data


#### Load df_jobs — IndoTech-Job Dataset


In [65]:
# Deteksi root proyek secara aman meskipun notebook dijalankan dari folder berbeda
cwd = Path.cwd()
job_rel_path = Path('data/raw/job_listings_dataset_v3.csv')
job_candidates = [cwd / job_rel_path, cwd.parent / job_rel_path, Path('D:/BisaKerja') / job_rel_path]
jobs_path = next((p for p in job_candidates if p.exists()), None)

if jobs_path is None:
    raise FileNotFoundError(
        f"File tidak ditemukan: {job_rel_path}. CWD aktif: {cwd}. "
        f"Coba pastikan notebook dijalankan dari root proyek atau file memang ada di data/raw/."
    )

PROJECT_ROOT = jobs_path.parents[2]
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

df_jobs = pd.read_csv(jobs_path)

# Seleksi kolom relevan — termasuk kolom kualitas untuk training AI
COLS_JOBS = [
    'job_id', 'company_name', 'title', 'normalized_title', 'category',
    'description', 'requirement_summary', 'work_type', 'employment_type',
    'experience_level', 'province', 'city', 'salary_min', 'salary_max',
    'salary_currency', 'salary_display', 'skills_top_10_names',
    'requirements_concat', 'language_signal', 'source_posted_at', 'status',
    # Kolom kualitas untuk quality gate training
    'fit_input_quality_score', 'fit_input_has_requirements',
    'fit_input_has_skills', 'skills_count_total', 'description_length_chars',
]
df_jobs = df_jobs[COLS_JOBS].copy()

print(f'File jobs : {jobs_path}')
print(f'Shape     : {df_jobs.shape[0]:,} baris x {df_jobs.shape[1]} kolom')
print(f'Kolom     : {df_jobs.columns.tolist()}')
df_jobs.head(5)


File jobs : d:\BisaKerja\data\raw\job_listings_dataset_v3.csv
Shape     : 3,883 baris x 26 kolom
Kolom     : ['job_id', 'company_name', 'title', 'normalized_title', 'category', 'description', 'requirement_summary', 'work_type', 'employment_type', 'experience_level', 'province', 'city', 'salary_min', 'salary_max', 'salary_currency', 'salary_display', 'skills_top_10_names', 'requirements_concat', 'language_signal', 'source_posted_at', 'status', 'fit_input_quality_score', 'fit_input_has_requirements', 'fit_input_has_skills', 'skills_count_total', 'description_length_chars']


,job_id,company_name,title,normalized_title,category,description,requirement_summary,work_type,employment_type,experience_level,province,city,salary_min,salary_max,salary_currency,salary_display,skills_top_10_names,requirements_concat,language_signal,source_posted_at,status,fit_input_quality_score,fit_input_has_requirements,fit_input_has_skills,skills_count_total,description_length_chars
0,313b29c7-3110-4816-a160-82d9ac5bcc8e,IDX Technology Information Solutions,System Developer,System Developer,Information Technology & Communications,"<p>IDX Technology Information Solutions is seeking a System Developer to build scalable systems,...",<ul><li>Build scalable systems</li><li>Support digital products</li></ul>,ONSITE,CONTRACT,ENTRY_LEVEL,Jakarta Raya,Jakarta Selatan,NaN,NaN,IDR,Not specified,API Development | System Development,Build scalable systems || Support digital products,EN,2026-04-20T11:00:19Z,ACTIVE,100,1,1,2,185
1,85dde9be-5e85-450f-a261-3ca1e6a0093d,Etex Building Performance Indonesia,IT Regional Support Engineer APAC/Indonesia,IT Regional Support Engineer APAC/Indonesia,Information Technology & Communications,<p>Etex Building Performance Indonesia is recruiting for an IT Regional Support Engineer APAC/In...,<p>Provide IT support.</p>,ONSITE,FULL_TIME,ENTRY_LEVEL,Jawa Timur,Gresik,NaN,NaN,IDR,Not specified,Regional Support | IT Support,Provide IT support,EN,2026-05-05T03:54:17Z,ACTIVE,100,1,1,2,215
2,bc76e742-3e2b-48b0-bb8a-1ae1976e0f1a,Precision Installations,Accountant/Financial Accountant (APAC),Financial Accountant,Accounting & Finance,<p>DPI is shaping the future of digital infrastructure in APAC. come and join us! DPI is proud t...,<p>The role requires accounting and financial reporting expertise.</p>,ONSITE,CONTRACT,ENTRY_LEVEL,Kepulauan Riau,Batam,NaN,NaN,IDR,Not specified,Financial Reporting | Accounting,The role requires accounting and financial reporting expertise,EN,2026-04-30T15:00:11Z,ACTIVE,100,1,1,2,384
3,b24553f8-4c1e-4956-952e-f955e7e30d49,Asia Technology Intersolutions,DevOps,DevOps Engineer,Information Technology,"<p>The DevOps role at Asia Technology Intersolutions in Bandung, Jawa Barat focuses on core cont...",<p>experience in DevOps.</p>,ONSITE,FULL_TIME,ENTRY_LEVEL,Jawa Barat,Bandung,4000000.0,6000000.0,IDR,"IDR 4,000,000 - 6,000,000 / month",DevOps,experience in DevOps,EN,2026-04-27T02:36:48Z,ACTIVE,100,1,1,1,211
4,02ae1d5e-af4a-4c28-b81f-7bca5d2b502f,Kamoro Maxima Integra,System Administrator,System Administrator,Information Technology,"<p>The System Administrator role at Kamoro Maxima Integra in Manyar, Jawa Timur focuses on core ...",<p>3 years of work experience as a System Administrator.</p>,ONSITE,FULL_TIME,ENTRY_LEVEL,Jawa Timur,Manyar,NaN,NaN,IDR,Not specified,System Administration,3 years of work experience as a System Administrator,EN,2026-05-12T04:56:59Z,ACTIVE,100,1,1,1,247


**Insight:**
- Dataset IndoTech-Job memiliki 2.099 baris dan 26 kolom relevan hasil seleksi dari 126 kolom asli.
- Kolom `skills_top_10_names` dan `requirements_concat` adalah kunci utama untuk analisis skill gap.
- Kolom `fit_input_quality_score`, `fit_input_has_requirements`, `fit_input_has_skills`, dan `skills_count_total` digunakan sebagai quality gate untuk dataset training model AI.


#### Load df_resume — TechTalent-Profile Dataset


In [66]:
resume_path = RAW_DIR / 'resume_dataset.csv'

if not resume_path.exists():
    raise FileNotFoundError(f'File resume tidak ditemukan: {resume_path}')

df_resume = pd.read_csv(resume_path)

print(f'File resume: {resume_path}')
print(f'Shape      : {df_resume.shape[0]:,} baris x {df_resume.shape[1]} kolom')
print(f'Kolom      : {df_resume.columns.tolist()}')
df_resume.head(3)


File resume: d:\BisaKerja\data\raw\resume_dataset.csv
Shape      : 100,000 baris x 8 kolom
Kolom      : ['ID', 'Name', 'Skills', 'Projects', 'Education', 'Experience', 'Job_Role', 'Required_Skills']


,ID,Name,Skills,Projects,Education,Experience,Job_Role,Required_Skills
0,1,Aarav Patel,"HTML, JavaScript, Node.js, Manufacturing, SIEM, Electronics, Docker, Surgery","Healthcare App, AI Chatbot",B.Tech,Fresher,Web Developer,"HTML, CSS, JavaScript, React, Node.js"
1,2,Rohan Iyer,"APIs, Machine Learning, CSS, Visualization, PyTorch, Python, Network Security, CAD","Resume Analyzer, IoT Smart Home",Diploma,Fresher,ML Engineer,"Python, Deep Learning, TensorFlow, PyTorch, Machine Learning"
2,3,Neha Gupta,"React, Medical Knowledge, APIs, Python, Manufacturing, Git, Embedded Systems, RTOS","Stock Prediction Model, Cybersecurity Monitoring Tool",MBA,3-5 years,Backend Developer,"Python, Java, SQL, APIs, Git"


**Insight:**
- Dataset TechTalent-Profile memiliki 100.000 baris dan 8 kolom.
- Kolom tersedia: `ID`, `Name`, `Skills`, `Projects`, `Education`, `Experience`, `Job_Role`, `Required_Skills`.
- Dataset mencakup 10 job role dan 4 level pengalaman (Fresher, 1-2 years, 3-5 years, 5+ years).
- Catatan: Education menggunakan sistem India (B.Tech, M.Tech, MBBS, dll.) — akan di-mapping ke format Indonesia (S1, S2, D3) pada tahap cleaning.


### Assessing Data


#### 1. Gambaran Umum Dataset


In [67]:
print('=' * 55)
print('GAMBARAN UMUM — df_jobs')
print('=' * 55)
print(f'Jumlah baris : {df_jobs.shape[0]:,}')
print(f'Jumlah kolom : {df_jobs.shape[1]}')
print(f'\nTipe data:')
print(df_jobs.dtypes)
df_jobs.describe(include='all').T


GAMBARAN UMUM — df_jobs
Jumlah baris : 3,883
Jumlah kolom : 26

Tipe data:
job_id                         object
company_name                   object
title                          object
normalized_title               object
category                       object
description                    object
requirement_summary            object
work_type                      object
employment_type                object
experience_level               object
province                       object
city                           object
salary_min                    float64
salary_max                    float64
salary_currency                object
salary_display                 object
skills_top_10_names            object
requirements_concat            object
language_signal                object
source_posted_at               object
status                         object
fit_input_quality_score         int64
fit_input_has_requirements      int64
fit_input_has_skills            int64
skills_count_

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
job_id,3883,3883,313b29c7-3110-4816-a160-82d9ac5bcc8e,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_name,3883,2311,Pengiklan Anonim,153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,3883,2818,Data Analyst,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
normalized_title,3883,2461,Data Analyst,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,3811,298,Teknologi Informasi & Komunikasi,731,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,3883,3823,<p>Dealls is Indonesia's largest job portal &amp; mentoring platform. We help people easily find...,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
requirement_summary,3883,3200,"<p>Proficient in EXPIRES_SOON, Akan segera berakhir.</p>",232,NaN,NaN,NaN,NaN,NaN,NaN,NaN
work_type,3883,3,ONSITE,3505,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_type,3883,5,FULL_TIME,2995,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience_level,3883,5,ENTRY_LEVEL,2177,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
print('=' * 55)
print('GAMBARAN UMUM - df_resume')
print('=' * 55)
print(f'Jumlah baris : {df_resume.shape[0]:,}')
print(f'Jumlah kolom : {df_resume.shape[1]}')
print(f'\nTipe data:')
print(df_resume.dtypes)
df_resume.describe(include='all').T


GAMBARAN UMUM - df_resume
Jumlah baris : 100,000
Jumlah kolom : 8

Tipe data:
ID                  int64
Name               object
Skills             object
Projects           object
Education          object
Experience         object
Job_Role           object
Required_Skills    object
dtype: object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,100000.0,NaN,NaN,NaN,50000.5,28867.657797,1.0,25000.75,50000.5,75000.25,100000.0
Name,100000,180,Aditya Sharma,608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Skills,100000,99890,"Kubernetes, Cryptography, JavaScript, CSS, Node.js, Electronics, HTML, SolidWorks",3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Projects,100000,90,"E-commerce Website, Cloud Deployment Project",1197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Education,100000,7,Diploma,14388,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Experience,100000,4,Fresher,25084,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Job_Role,100000,10,Mechanical Engineer,10089,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Required_Skills,100000,10,"CAD, SolidWorks, Thermodynamics, Manufacturing, Design",10089,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 2. Identifikasi Missing Values


In [69]:
print('Missing Values - df_jobs')
print('-' * 45)
mv_jobs = pd.DataFrame({
    'Missing Count': df_jobs.isnull().sum(),
    'Missing (%)': (df_jobs.isnull().sum() / len(df_jobs) * 100).round(2)
}).query('`Missing Count` > 0')
print(mv_jobs.to_string())

print(f'\nMissing Values - df_resume')
print('-' * 45)
mv_resume = pd.DataFrame({
    'Missing Count': df_resume.isnull().sum(),
    'Missing (%)': (df_resume.isnull().sum() / len(df_resume) * 100).round(2)
}).query('`Missing Count` > 0')
if mv_resume.empty:
    print('Tidak ada missing value pada df_resume.')
else:
    print(mv_resume.to_string())


Missing Values - df_jobs
---------------------------------------------
            Missing Count  Missing (%)
category               72         1.85
province               34         0.88
salary_min           2410        62.07
salary_max           2404        61.91

Missing Values - df_resume
---------------------------------------------
Tidak ada missing value pada df_resume.


**Steps to Take:**
- `category` pada df_jobs memiliki ~835 missing value (39.8%) — akan ditangani dengan multi-signal tech detection (category + title + skills), bukan hanya exact matching.
- `salary_min` dan `salary_max` memiliki ~55% missing value — isi dengan `0` dan tandai dengan flag `has_salary_info`.
- `source_posted_at` memiliki missing value — isi dengan nilai median tanggal posting.
- `province` memiliki beberapa missing value — isi dengan `'Nasional'` sebagai nilai default.


#### 3. Identifikasi Duplicate Data


In [70]:
print('Duplikasi — df_jobs')
print('-' * 45)
print(f'Duplikasi baris penuh       : {df_jobs.duplicated().sum()}')
print(f'Duplikasi job_id            : {df_jobs["job_id"].duplicated().sum()}')
print(f'Duplikasi title + company   : {df_jobs.duplicated(subset=["title","company_name"]).sum()}')

dup_mask = df_jobs.duplicated(subset=['title','company_name'], keep=False)
print(f'\nSample duplikasi title+company (6 baris):')
print(df_jobs[dup_mask][['job_id','title','company_name']].head(6).to_string())

print(f'\nDuplikasi — df_resume')
print('-' * 45)
print(f'Duplikasi baris penuh : {df_resume.duplicated().sum()}')
print(f'Duplikasi ID          : {df_resume["ID"].duplicated().sum()}')
print(f'Duplikasi Name        : {df_resume["Name"].duplicated().sum():,} ({df_resume["Name"].duplicated().sum()/len(df_resume)*100:.1f}%)')
print(f'Duplikasi Skills      : {df_resume["Skills"].duplicated().sum():,}')


Duplikasi — df_jobs
---------------------------------------------
Duplikasi baris penuh       : 0
Duplikasi job_id            : 0
Duplikasi title + company   : 81

Sample duplikasi title+company (6 baris):
                                   job_id                                     title            company_name
27   56550d44-3f82-4e4a-86e5-bf1ff3f35b18                             Brand Manager        Pengiklan Anonim
137  0d2f8372-b844-43a0-8024-2edb3c254285                    Data Center Technician               Microsoft
261  3996351f-2871-46c4-bebd-2c550d19b17f                              Data Analyst        Pengiklan Anonim
268  45170fec-8f6e-41dc-b85f-2978102b65aa                   Data Analytic (Tableau)  Lawencon International
344  eba8732d-8d71-41e6-ace3-cbf0f92bbf16              Technical Consultant Analyst               Metrodata
371  17df209e-872c-4b0a-a274-1307307b6270  Business Consultant — Printing Solutions    Laysander Technology

Duplikasi — df_resume
---------------

**Steps to Take:**
- df_jobs: deduplikasi berdasarkan `job_id` terlebih dahulu, lalu `title + company_name` — pertahankan job_id terbesar (posting terbaru).
- df_resume: kolom `Name` memiliki 99.8% duplikasi — tidak informatif, perlu di-drop.
- df_resume: baris dengan kombinasi `Skills` identik perlu di-drop untuk menghindari bias model.


#### 4. Identifikasi Invalid Values


In [71]:
print('Invalid Values — df_jobs')
print('-' * 45)

for col in ['experience_level','work_type','employment_type','language_signal','status']:
    print(f'  {col:25}: {df_jobs[col].unique().tolist()}')

sal_one = df_jobs[df_jobs['salary_min'] == 1]
print(f'  salary_min = 1 (tidak wajar): {len(sal_one)} baris')

sal_valid = df_jobs.dropna(subset=['salary_min','salary_max'])
bad_sal = sal_valid[sal_valid['salary_min'] > sal_valid['salary_max']]
print(f'  salary_min > salary_max: {len(bad_sal)} baris')

has_html_desc = df_jobs['description'].str.contains('<[^>]+>', regex=True, na=False).sum()
has_html_req  = df_jobs['requirement_summary'].str.contains('<[^>]+>', regex=True, na=False).sum()
print(f'\n  description ber-HTML        : {has_html_desc} dari {len(df_jobs)} baris')
print(f'  requirement_summary ber-HTML: {has_html_req} dari {len(df_jobs)} baris')

print(f'\nInvalid Values — df_resume')
print('-' * 45)
for col in ['Job_Role','Experience','Education']:
    print(f'  {col:20}: {df_resume[col].unique().tolist()}')

edu_irrelevant = ['B.Tech','M.Tech','MBBS','B.Sc','M.Sc','MBA']
cnt = df_resume[df_resume['Education'].isin(edu_irrelevant)].shape[0]
print(f'\n  Education India-centric: {cnt:,} baris ({cnt/len(df_resume)*100:.1f}%) — akan di-mapping ke S1/S2/D3')


Invalid Values — df_jobs
---------------------------------------------
  experience_level         : ['ENTRY_LEVEL', 'LEAD', 'SENIOR', 'JUNIOR', 'MID_LEVEL']
  work_type                : ['ONSITE', 'HYBRID', 'REMOTE']
  employment_type          : ['CONTRACT', 'FULL_TIME', 'PART_TIME', 'INTERNSHIP', 'FREELANCE']
  language_signal          : ['EN', 'UNKNOWN', 'ID', 'MIXED']
  status                   : ['ACTIVE', 'EXPIRED']
  salary_min = 1 (tidak wajar): 0 baris
  salary_min > salary_max: 0 baris

  description ber-HTML        : 3883 dari 3883 baris
  requirement_summary ber-HTML: 3883 dari 3883 baris

Invalid Values — df_resume
---------------------------------------------
  Job_Role            : ['Web Developer', 'ML Engineer', 'Backend Developer', 'Data Scientist', 'Cloud Engineer', 'Cybersecurity Analyst', 'Doctor', 'Mechanical Engineer', 'Embedded Engineer', 'Business Analyst']
  Experience          : ['Fresher', '3-5 years', '1-2 years', '5+ years']
  Education           : ['B.Tech

**Steps to Take:**
- Lowongan dengan `status = EXPIRED` (4 baris) dan `language_signal = UNKNOWN` perlu di-filter.
- `salary_min = 1` (tidak wajar secara bisnis) di-treat sebagai missing value.
- Seluruh `description` dan `requirement_summary` mengandung HTML tag — perlu di-strip menggunakan BeautifulSoup.
- `Education` di df_resume menggunakan sistem India — di-mapping ke format Indonesia (S1, S2, D3).


#### 5. Identifikasi Inconsistent Values


In [72]:
print('Inconsistent Values — df_jobs')
print('-' * 45)

print('Distribusi Category (sebelum normalisasi — top 30):')
print(df_jobs['category'].value_counts(dropna=False).head(30).to_string())
print(f'\nTotal variasi kategori unik: {df_jobs["category"].nunique()}')
print(f'Kategori null              : {df_jobs["category"].isna().sum()} ({df_jobs["category"].isna().sum()/len(df_jobs)*100:.1f}%)')

pipe_sep  = df_jobs['skills_top_10_names'].str.contains('|', regex=False, na=False).sum()
comma_sep = df_jobs['skills_top_10_names'].str.contains(',', regex=False, na=False).sum()
print(f'\nSeparator skills_top_10_names:')
print(f'  Pipe  (|) : {pipe_sep} baris')
print(f'  Koma  (,) : {comma_sep} baris')

print(f'\nInconsistent Values — df_resume')
print('-' * 45)
comma_r = df_resume['Skills'].str.contains(',', regex=False, na=False).sum()
print(f'  Skills separator koma: {comma_r:,} baris')


Inconsistent Values — df_jobs
---------------------------------------------
Distribusi Category (sebelum normalisasi — top 30):
category
Teknologi Informasi & Komunikasi           731
Technology                                 219
Penjualan                                  160
Pemasaran & Komunikasi                     149
Information Technology & Communications    148
IT and Software                            141
Manufaktur, Transportasi & Logistik        140
Sales                                       95
Information Technology & Communication      85
Sains & Teknologi                           76
Operation & Technical Support               76
Teknik                                      72
NaN                                         72
Administrasi & Dukungan Perkantoran         68
Computer & Software                         61
Ritel & Produk Konsumen                     43
Desain & Arsitektur                         40
Developer/Programmer                        40
Marketing & Commu

**Steps to Take:**
- `category` memiliki 204 variasi unik dengan 39.8% null. Pendekatan exact matching `.isin()` akan gagal menangkap sebagian besar data tech. Solusi: multi-signal detection (category regex + title keyword + skills keyword).
- `skills_top_10_names` menggunakan separator `|` — perlu diubah ke `,` agar konsisten dengan df_resume.


#### 6. Identifikasi Irrelevant Data


In [73]:
print('Irrelevant Data — df_jobs (preview sebelum multi-signal detection)')
print('-' * 45)

# Preview: berapa banyak null-category yang sebenarnya adalah tech job
null_cat = df_jobs[df_jobs['category'].isna()].copy()
TECH_TITLE_PREVIEW = [
    'software','developer','engineer','data','analyst','backend','frontend',
    'fullstack','full stack','devops','cloud','machine learning','deep learning',
    'artificial intelligence','ai','ml','cybersecurity','network','database',
    'qa','quality assurance','product manager','it support','it ','golang',
    'python','java','react','mobile','android','ios','flutter','ui/ux','nlp',
]
null_cat['looks_tech'] = null_cat['title'].str.lower().apply(
    lambda t: any(kw in str(t) for kw in TECH_TITLE_PREVIEW)
)
print(f'  Null-category rows         : {len(null_cat)}')
print(f'  Di antaranya terlihat tech  : {null_cat["looks_tech"].sum()} baris')
print(f'  (Ini yang akan hilang jika exact matching digunakan)')

print(f'\nIrrelevant Data — df_resume (Job Role Non-Tech)')
print('-' * 45)
NON_TECH_ROLES = ['Doctor', 'Mechanical Engineer', 'Embedded Engineer']
nontech_resume = df_resume[df_resume['Job_Role'].isin(NON_TECH_ROLES)].shape[0]
print(f'  Job role non-tech: {nontech_resume:,} baris ({nontech_resume/len(df_resume)*100:.1f}%)')
print(df_resume[df_resume['Job_Role'].isin(NON_TECH_ROLES)]['Job_Role'].value_counts().to_string())


Irrelevant Data — df_jobs (preview sebelum multi-signal detection)
---------------------------------------------
  Null-category rows         : 72
  Di antaranya terlihat tech  : 40 baris
  (Ini yang akan hilang jika exact matching digunakan)

Irrelevant Data — df_resume (Job Role Non-Tech)
---------------------------------------------
  Job role non-tech: 29,995 baris (30.0%)
Job_Role
Mechanical Engineer    10089
Doctor                  9980
Embedded Engineer       9926


**Steps to Take:**
- df_jobs: gunakan multi-signal tech detection (category regex OR title keyword) untuk menangkap semua job tech termasuk yang null-category. Simpan `is_tech` dan `tech_signal_source` sebagai kolom audit.
- df_resume: hapus baris dengan job role non-tech (`Doctor`, `Mechanical Engineer`, `Embedded Engineer`).


#### 7. Identifikasi Requirement Summary Terlalu Pendek


In [74]:
print('Requirement Summary Terlalu Pendek — df_jobs')
print('-' * 45)

df_jobs['req_len'] = df_jobs['requirement_summary'].str.len().fillna(0)
short_req = df_jobs[df_jobs['req_len'] < 50]
print(f'  requirement_summary < 50 karakter: {len(short_req)} baris')
print(f'\n  Statistik panjang requirement_summary:')
print(df_jobs['req_len'].describe().round(0))


Requirement Summary Terlalu Pendek — df_jobs
---------------------------------------------
  requirement_summary < 50 karakter: 745 baris

  Statistik panjang requirement_summary:
count    3883.0
mean      119.0
std       114.0
min        11.0
25%        56.0
50%        83.0
75%       135.0
max       791.0
Name: req_len, dtype: float64


**Steps to Take:**
- Baris dengan `requirement_summary` kurang dari 50 karakter informasinya tidak cukup untuk model — perlu di-filter setelah tahap tech filtering agar tidak membuang data yang masih bisa dipakai untuk EDA.


### Cleaning Data


#### Fix 1: Hapus Lowongan EXPIRED & Language UNKNOWN — df_jobs


In [75]:
# before = len(df_jobs)

# df_jobs = df_jobs[df_jobs['status'] == 'ACTIVE'].copy()
# df_jobs = df_jobs[df_jobs['language_signal'] != 'UNKNOWN'].copy()

# after = len(df_jobs)
# print(f'Filter EXPIRED & UNKNOWN:')
# print(f'  Sebelum : {before:,} | Sesudah : {after:,} | Dihapus : {before - after:,}')


#### Fix 2: Handle Duplicate — df_jobs


In [76]:
before = len(df_jobs)

# Pertahankan job_id terbesar (posting terbaru)
df_jobs = df_jobs.sort_values('job_id', ascending=False)
df_jobs = df_jobs.drop_duplicates(subset=['job_id'], keep='first').copy()
df_jobs = df_jobs.drop_duplicates(subset=['title', 'company_name'], keep='first').copy()

after = len(df_jobs)
print(f'Drop duplikasi job_id + title+company:')
print(f'  Sebelum : {before:,} | Sesudah : {after:,} | Dihapus : {before - after:,}')


Drop duplikasi job_id + title+company:
  Sebelum : 3,883 | Sesudah : 3,802 | Dihapus : 81


#### Fix 3: Strip HTML Tags — df_jobs


In [77]:
def strip_html(text):
    """Menghapus semua tag HTML dan membersihkan whitespace berlebih."""
    if pd.isna(text):
        return ''
    soup = BeautifulSoup(str(text), 'html.parser')
    clean = soup.get_text(separator=' ')
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

df_jobs['description']         = df_jobs['description'].apply(strip_html)
df_jobs['requirement_summary'] = df_jobs['requirement_summary'].apply(strip_html)

html_remaining = df_jobs['description'].str.contains('<[^>]+>', regex=True, na=False).sum()
print(f'Strip HTML selesai.')
print(f'  HTML tersisa di description: {html_remaining}')
print(f'\n  Contoh description bersih:')
print(df_jobs['description'].iloc[0][:300])


Strip HTML selesai.
  HTML tersisa di description: 0

  Contoh description bersih:
Why You Should Choose This Position With Us? 1. Impactful Role, Supportive & Growing Work Environment, Diverse Tasks.


#### Fix 4: Normalisasi Category + Multi-Signal Tech Detection — df_jobs


In [78]:
# CATEGORY_MAPPING: unifikasi 204 variasi ke kategori standar
CATEGORY_MAPPING = {
    # Tech — IT & Software
    'IT and Software'                       : 'IT & Software',
    'IT & Engineering'                      : 'IT & Software',
    'IT & Software Engineering'             : 'IT & Software',
    'IT/Software Engineering'               : 'IT & Software',
    'IT & Software'                         : 'IT & Software',
    'Engineering (IT/Software)'             : 'IT & Software',
    'Teknologi Informasi'                   : 'IT & Software',
    'Teknologi Informasi & Komunikasi'      : 'IT & Software',
    'Teknologi'                             : 'IT & Software',
    'Tech Company, Fintech'                 : 'IT & Software',
    'Developer/Programmer'                  : 'IT & Software',
    'Pengembang Perangkat Lunak'            : 'IT & Software',
    'IT and Software Engineering'           : 'IT & Software',
    'Computer & Software'                   : 'IT & Software',
    'Computer Software'                     : 'IT & Software',
    'Information Technology and Services'   : 'IT & Software',
    'Full Stack Developer'                  : 'IT & Software',
    'Frontend Developer'                    : 'IT & Software',
    'Backend Developer'                     : 'IT & Software',
    'Backend Development'                   : 'IT & Software',
    'Mobile Developer'                      : 'IT & Software',
    'Devops Engineer'                       : 'IT & Software',
    'AI Engineer'                           : 'IT & Software',
    'Network Engineer'                      : 'IT & Software',
    'Database Administrator'                : 'IT & Software',
    'System Analyst'                        : 'IT & Software',
    'Game Developer'                        : 'IT & Software',
    'Software Architect'                    : 'IT & Software',
    'System Security'                       : 'IT & Software',
    'Android Developer'                     : 'IT & Software',
    'IT Support'                            : 'IT & Software',
    'Cybersecurity'                         : 'IT & Software',
    # Tech — Data & Analytics
    'Data & Product'                        : 'Data & Analytics',
    'Data Science & Machine Learning'       : 'Data & Analytics',
    'Data Analyst'                          : 'Data & Analytics',
    'Data Engineer'                         : 'Data & Analytics',
    'Data Scientist'                        : 'Data & Analytics',
    'Data'                                  : 'Data & Analytics',
    'Data & Analytics'                      : 'Data & Analytics',
    # Tech — Product Management
    'Product Management & Project Management': 'Product Management',
    'Product Management'                    : 'Product Management',
    'Product Manager'                       : 'Product Management',
    'Product – Software'                     : 'Product Management',
    'Product - Software'                    : 'Product Management',
    # Tech — QA & Testing
    'QA Engineer'                           : 'QA & Testing',
    'QA & Testing'                          : 'QA & Testing',
    'QA & Quality Control'                  : 'QA & Testing',
    'Quality Control'                       : 'QA & Testing',
    'Software Quality Assurance'            : 'QA & Testing',
    'Uji Coba & Penjaminan Mutu'            : 'QA & Testing',
    'Penjaminan & Kontrol Mutu'             : 'QA & Testing',
    # Tech — Design & UX
    'UI/UX Designer'                        : 'Design & UX',
    'Visual & Interactive Design'           : 'Design & UX',
    'Desain Web & Interaksi'                : 'Design & UX',
    # Non-tech (tetap disimpan untuk df_nontech)
    'Sales and Marketing'                   : 'Sales & Marketing',
    'Penjualan'                             : 'Sales & Marketing',
    'Sales'                                 : 'Sales & Marketing',
    'Marketing'                             : 'Sales & Marketing',
    'Pemasaran'                             : 'Sales & Marketing',
    'Accounting and Finance'                : 'Finance & Accounting',
    'Akuntansi'                             : 'Finance & Accounting',
    'Finance'                               : 'Finance & Accounting',
    'Human Resources'                       : 'HR & Recruitment',
    'Sumber Daya Manusia'                   : 'HR & Recruitment',
    'Administrasi'                          : 'Administration',
    'Admin'                                 : 'Administration',
    'Administration and Coordination'       : 'Administration',
    'Retail'                                : 'Retail & FMCG',
    'Food & Beverage'                       : 'Hospitality',
    'Hospitality and Tourism'               : 'Hospitality',
    'Health and Medical'                    : 'Healthcare',
    'Healthcare'                            : 'Healthcare',
    'Education and Training'                : 'Education',
    'Architecture and Engineering'          : 'Engineering',
    'Teknik'                                : 'Engineering',
    'Media and Creatives'                   : 'Media & Creative',
    'Desain'                                : 'Media & Creative',
    'Customer Service'                      : 'Customer Service',
    'Lain-lain'                             : 'Others',
    'Others'                                : 'Others',
}

# Terapkan mapping — HARUS sebelum filtering
df_jobs['category'] = df_jobs['category'].replace(CATEGORY_MAPPING)

# ── Set kategori tech final setelah mapping
TECH_CATS_FINAL = {
    'IT & Software', 'Data & Analytics', 'Product Management',
    'QA & Testing', 'Design & UX',
}

# ── Signal A: category-based (regex untuk tangkap sisa variasi) 
TECH_CAT_PATTERNS = [
    r'teknologi informasi', r'it and software', r'it & software',
    r'it & engineering', r'it support', r'computer & software',
    r'full.?stack', r'frontend', r'backend', r'mobile developer',
    r'data & analytics', r'data & product', r'data analyst',
    r'data engineer', r'data scientist', r'devops', r'cybersecurity',
    r'ai engineer', r'product management', r'product manager',
    r'qa engineer', r'qa & test', r'quality assurance',
    r'database', r'network engineer', r'system analyst',
    r'software architect', r'ui/ux', r'visual & interactive',
    r'developer.programmer', r'pengembang perangkat',
    r'sains & teknologi', r'business analyst', r'automation engineer',
    r'uji coba',
]

# ── Signal B: title-based (menangkap null-category tech jobs) 
TECH_TITLE_PATTERNS = [
    r'\bsoftware\b', r'\bdeveloper\b', r'\bprogrammer\b',
    r'data scientist', r'data analyst', r'data engineer',
    r'\bbackend\b', r'\bfrontend\b', r'full.?stack', r'\bdevops\b',
    r'\bcloud\b', r'machine learning', r'deep learning',
    r'artificial intelligence', r'\bai\b', r'\bml\b',
    r'\bcybersecurity\b', r'quality assurance', r'\bqa\b',
    r'mobile developer', r'flutter', r'ui/ux', r'system analyst',
    r'it support', r'it infrastructure', r'\bgolang\b',
    r'prompt engineer', r'tech lead', r'solution architect',
    r'game developer', r'\bnlp\b', r'network engineer',
    r'database admin', r'android developer', r'software engineer',
]

# ── Signal C: skills-based (tiebreaker, minimal 2 skill tech) 
TECH_SKILL_KEYWORDS = [
    'python', 'javascript', 'typescript', 'java', 'golang', 'react',
    'node.js', 'tensorflow', 'pytorch', 'sql', 'docker', 'kubernetes',
    'aws', 'gcp', 'azure', 'machine learning', 'deep learning',
    'flutter', 'kotlin', 'postgresql', 'php',
]

def is_tech_cat(cat):
    if pd.isna(cat): return False
    if cat in TECH_CATS_FINAL: return True
    c = cat.lower()
    return any(re.search(p, c) for p in TECH_CAT_PATTERNS)

def is_tech_title(title):
    if pd.isna(title): return False
    t = title.lower()
    return any(re.search(p, t) for p in TECH_TITLE_PATTERNS)

def is_tech_skills(skills):
    if pd.isna(skills): return False
    s = skills.lower()
    return sum(1 for kw in TECH_SKILL_KEYWORDS if kw in s) >= 2

# Terapkan semua signal
df_jobs['_is_tech_cat']    = df_jobs['category'].apply(is_tech_cat)
df_jobs['_is_tech_title']  = df_jobs['normalized_title'].apply(is_tech_title)
df_jobs['_is_tech_skills'] = df_jobs['skills_top_10_names'].apply(is_tech_skills)

# is_tech = category OR title (skills hanya untuk audit)
df_jobs['is_tech'] = df_jobs['_is_tech_cat'] | df_jobs['_is_tech_title']

# Kolom audit: dari mana sinyal tech berasal
def get_tech_signal_source(row):
    sources = []
    if row['_is_tech_cat']:    sources.append('category')
    if row['_is_tech_title']:  sources.append('title')
    if row['_is_tech_skills']: sources.append('skills')
    return '|'.join(sources) if sources else 'none'

df_jobs['tech_signal_source'] = df_jobs.apply(get_tech_signal_source, axis=1)

# Split — df_nontech JANGAN di-drop, simpan untuk referensi
df_tech    = df_jobs[df_jobs['is_tech']].copy()
df_nontech = df_jobs[~df_jobs['is_tech']].copy()

print(f'Multi-signal tech detection selesai:')
print(f'  Tech jobs    : {len(df_tech):,}')
print(f'  Non-tech jobs: {len(df_nontech):,}')
print(f'\nDistribusi tech_signal_source:')
print(df_tech['tech_signal_source'].value_counts().to_string())
print(f'\nDistribusi category (df_tech):')
print(df_tech['category'].value_counts().to_string())

# Lanjutkan pipeline dengan df_tech sebagai df_jobs utama
df_jobs = df_tech.copy()


Multi-signal tech detection selesai:
  Tech jobs    : 2,073
  Non-tech jobs: 1,729

Distribusi tech_signal_source:
tech_signal_source
category|title           629
category                 552
title                    523
category|title|skills    272
title|skills              66
category|skills           31

Distribusi category (df_tech):
category
IT & Software                                1194
Technology                                    168
Sains & Teknologi                              74
Information Technology & Communications        69
Operation & Technical Support                  56
Product Management                             53
Data & Analytics                               51
Information Technology & Communication         46
QA & Testing                                   45
Manufaktur, Transportasi & Logistik            38
Frontend/Mobile Development                    25
Design                                         22
Design & UX                                    19
I

#### Fix 4b: Inferensi Category dari normalized_title — Null Category yang Tersisa

Setelah multi-signal detection, baris yang masuk via `title` signal namun tidak memiliki label category di data asli masih bernilai `null`. Semua baris ini adalah tech jobs valid (DevOps, Full Stack, Data Engineer, QA, UI/UX, dll.) — perlu di-assign kategori berdasarkan `normalized_title`, dari yang paling spesifik sampai fallback `IT & Software`.

In [79]:
null_before = df_jobs['category'].isna().sum()
print(f'Null category sebelum Fix 4b: {null_before} baris')

def infer_category_from_title(title):
    """
    Inferensi kategori dari normalized_title untuk baris yang lolos
    via title signal tetapi tidak memiliki label category di data asli.
    Urutan cek: paling spesifik dulu, fallback ke IT & Software.
    """
    if pd.isna(title) or str(title).strip() == '':
        return 'IT & Software'
    t = title.lower()
    # Data & Analytics
    if any(x in t for x in ['data scientist', 'data science', 'data analyst',
                              'data engineer', 'machine learning', 'ml engineer',
                              'ai analyst', 'business intelligence']):
        return 'Data & Analytics'
    # QA & Testing
    if any(x in t for x in ['quality assurance', 'qa officer', 'qa engineer',
                              'qa junior', 'qa tester', 'qa analyst',
                              'tester', 'testing']):
        return 'QA & Testing'
    # Design & UX
    if any(x in t for x in ['ui/ux', 'ux designer', 'ux staff',
                              'ui ux', 'product designer', 'visual designer']):
        return 'Design & UX'
    # Product Management
    if any(x in t for x in ['product manager', 'product management', 'product owner']):
        return 'Product Management'
    # Default: DevOps, Full Stack, Backend, Network, IT Support, dll.
    return 'IT & Software'

# Terapkan hanya pada baris yang category-nya masih null
null_mask = df_jobs['category'].isna()
df_jobs.loc[null_mask, 'category'] = (
    df_jobs.loc[null_mask, 'normalized_title']
    .apply(infer_category_from_title)
)

null_after = df_jobs['category'].isna().sum()
print(f'Null category setelah Fix 4b  : {null_after} baris')
print(f'Baris yang berhasil di-assign  : {null_before - null_after} baris')
print(f'\nDistribusi category setelah Fix 4b:')
print(df_jobs['category'].value_counts().to_string())


Null category sebelum Fix 4b: 31 baris
Null category setelah Fix 4b  : 0 baris
Baris yang berhasil di-assign  : 31 baris

Distribusi category setelah Fix 4b:
category
IT & Software                                1212
Technology                                    168
Sains & Teknologi                              74
Information Technology & Communications        69
Operation & Technical Support                  56
Data & Analytics                               54
Product Management                             53
QA & Testing                                   49
Information Technology & Communication         46
Manufaktur, Transportasi & Logistik            38
Design & UX                                    25
Frontend/Mobile Development                    25
Design                                         22
Information Technology                         17
Business Analyst                               15
Desain & Arsitektur                             9
Science & Technology             

#### Fix 5: Normalisasi Province — df_jobs


In [80]:
PROVINCE_MAPPING = {
    'Daerah Khusus Ibukota Jakarta' : 'DKI Jakarta',
    'Central Java (Jawa Tengah)'    : 'Jawa Tengah',
    'West Java (Jawa Barat)'        : 'Jawa Barat',
    'East Java (Jawa Timur)'        : 'Jawa Timur',
    'North Sumatra (Sumatra Utara)' : 'Sumatera Utara',
    'Indonesia'                     : 'Nasional',
}
df_jobs['province'] = df_jobs['province'].replace(PROVINCE_MAPPING)
df_jobs['province'] = df_jobs['province'].fillna('Nasional')

print('Normalisasi province selesai.')
print(df_jobs['province'].value_counts().head(10).to_string())


Normalisasi province selesai.
province
Jakarta Raya     732
DKI Jakarta      407
Jawa Barat       191
Banten           173
Nasional         124
Jawa Timur       123
DI Yogyakarta     71
Bali              50
Jawa Tengah       42
Jakarta           20


#### Fix 6: Handle Missing Salary — df_jobs


In [81]:
df_jobs['has_salary_info'] = (~df_jobs['salary_min'].isna()).astype(int)

# Salary_min = 1 tidak wajar secara bisnis
df_jobs.loc[df_jobs['salary_min'] == 1, 'salary_min'] = np.nan

df_jobs['salary_min'] = df_jobs['salary_min'].fillna(0)
df_jobs['salary_max'] = df_jobs['salary_max'].fillna(0)

print(f'Handle salary selesai.')
print(f'  Lowongan dengan info gaji: {df_jobs["has_salary_info"].sum()} dari {len(df_jobs)}')


Handle salary selesai.
  Lowongan dengan info gaji: 844 dari 2073


#### Fix 7: Handle Missing source_posted_at — df_jobs


In [82]:
df_jobs['source_posted_at'] = pd.to_datetime(df_jobs['source_posted_at'], errors='coerce')
median_date = df_jobs['source_posted_at'].dropna().median()
df_jobs['source_posted_at'] = df_jobs['source_posted_at'].fillna(median_date)

print(f'Handle source_posted_at selesai.')
print(f'  Median date     : {median_date}')
print(f'  Missing tersisa : {df_jobs["source_posted_at"].isna().sum()}')


Handle source_posted_at selesai.
  Median date     : 2026-04-20 08:38:47.489999872+00:00
  Missing tersisa : 0


#### Fix 8: Standardisasi Separator Skills — df_jobs


In [83]:
def standardize_skills(skill_str, sep_in='|', sep_out=','):
    """Ubah separator pipe ke koma dan strip whitespace tiap skill."""
    if pd.isna(skill_str):
        return ''
    skills = [s.strip() for s in str(skill_str).split(sep_in)]
    skills = [s for s in skills if s]
    return f'{sep_out} '.join(skills)

df_jobs['skills_clean'] = df_jobs['skills_top_10_names'].apply(standardize_skills)

print('Standardisasi separator skills selesai.')
print('  Contoh (3 baris):')
for s in df_jobs['skills_clean'].head(3):
    print(f'    {s}')


Standardisasi separator skills selesai.
  Contoh (3 baris):
    Typescript, React, Javascript, CSS, Next.js, HTML, Tailwind, React.Js
    Gitlab, DevSecOps, CD, CI, CI/CD
    Power BI, Apache Spark, Tableau, Google BigQuery, Python, Kubernetes, Kafka, SQL, Data Analysis


#### Fix 9: Filter Requirement Summary Terlalu Pendek — df_jobs


In [84]:
# before = len(df_jobs)
# df_jobs = df_jobs[df_jobs['requirement_summary'].str.len() >= 50].copy()
# after  = len(df_jobs)

# # Hapus kolom helper internal
# df_jobs = df_jobs.drop(columns=['req_len', '_is_tech_cat', '_is_tech_title',
#                                   '_is_tech_skills'], errors='ignore')

# print(f'Filter requirement summary pendek:')
# print(f'  Sebelum : {before:,} | Sesudah : {after:,} | Dihapus : {before - after:,}')


#### Fix 10: Quality Gate untuk Dataset Training AI — df_jobs


In [85]:
# Tandai baris yang siap digunakan sebagai training data AI
# Kriteria: quality score >= 60, punya skill, deskripsi cukup panjang
df_jobs['is_train_ready'] = (
    (df_jobs['fit_input_quality_score'].fillna(0) >= 60) &
    (df_jobs['skills_count_total'].fillna(0) >= 1) &
    (df_jobs['description_length_chars'].fillna(0) >= 100)
).astype(int)

train_ready = df_jobs['is_train_ready'].sum()
print(f'Quality gate training data:')
print(f'  Total tech jobs  : {len(df_jobs):,}')
print(f'  Training-ready   : {train_ready:,} ({train_ready/len(df_jobs)*100:.1f}%)')
print(f'  EDA-only         : {len(df_jobs)-train_ready:,}')


Quality gate training data:
  Total tech jobs  : 2,073
  Training-ready   : 2,071 (99.9%)
  EDA-only         : 2


#### Fix 11: Filter Job Role Non-Tech — df_resume


In [86]:
NON_TECH_ROLES = ['Doctor', 'Mechanical Engineer', 'Embedded Engineer']
before = len(df_resume)
df_resume = df_resume[~df_resume['Job_Role'].isin(NON_TECH_ROLES)].copy()
after  = len(df_resume)

print(f'Filter job role non-tech:')
print(f'  Sebelum : {before:,} | Sesudah : {after:,} | Dihapus : {before - after:,}')
print(f'\n  Distribusi Job_Role setelah filter:')
print(df_resume['Job_Role'].value_counts().to_string())


Filter job role non-tech:
  Sebelum : 100,000 | Sesudah : 70,005 | Dihapus : 29,995

  Distribusi Job_Role setelah filter:
Job_Role
Cloud Engineer           10087
Data Scientist           10076
Cybersecurity Analyst    10041
Web Developer             9985
Business Analyst          9977
Backend Developer         9958
ML Engineer               9881


#### Fix 12: Drop Duplikasi Skills & Kolom Name — df_resume


In [87]:
before = len(df_resume)
df_resume = df_resume.drop_duplicates(subset=['Skills']).copy()
after  = len(df_resume)
print(f'Drop duplikasi Skills:')
print(f'  Sebelum : {before:,} | Sesudah : {after:,} | Dihapus : {before - after:,}')

df_resume = df_resume.drop(columns=['Name'], errors='ignore')
print(f'\nKolom Name di-drop (99.8% duplikat, tidak informatif).')
print(f'  Kolom tersisa: {df_resume.columns.tolist()}')


Drop duplikasi Skills:
  Sebelum : 70,005 | Sesudah : 69,929 | Dihapus : 76

Kolom Name di-drop (99.8% duplikat, tidak informatif).
  Kolom tersisa: ['ID', 'Skills', 'Projects', 'Education', 'Experience', 'Job_Role', 'Required_Skills']


#### Fix 13: Cleaning Skills & Required_Skills — df_resume


In [88]:
def clean_skills(skill_str):
    """Strip whitespace tiap skill dan normalisasi separator ke koma."""
    if pd.isna(skill_str):
        return ''
    skills = [s.strip() for s in str(skill_str).split(',')]
    return ', '.join([s for s in skills if s])

df_resume['Skills']          = df_resume['Skills'].apply(clean_skills)
df_resume['Required_Skills'] = df_resume['Required_Skills'].apply(clean_skills)

print('Cleaning skills selesai.')
print('  Contoh Skills (3 baris):')
for s in df_resume['Skills'].head(3):
    print(f'    {s}')


Cleaning skills selesai.
  Contoh Skills (3 baris):
    HTML, JavaScript, Node.js, Manufacturing, SIEM, Electronics, Docker, Surgery
    APIs, Machine Learning, CSS, Visualization, PyTorch, Python, Network Security, CAD
    React, Medical Knowledge, APIs, Python, Manufacturing, Git, Embedded Systems, RTOS


#### Fix 14: Mapping Education ke Format Indonesia — df_resume


In [89]:
EDU_MAPPING = {
    'B.Tech'  : 'S1',
    'B.Sc'    : 'S1',
    'MBBS'    : 'S1',
    'Diploma' : 'D3',
    'M.Tech'  : 'S2',
    'M.Sc'    : 'S2',
    'MBA'     : 'S2',
}
df_resume['Education'] = df_resume['Education'].replace(EDU_MAPPING)

print('Mapping education selesai.')
print(df_resume['Education'].value_counts().to_string())


Mapping education selesai.
Education
S1    30024
S2    29861
D3    10044


#### Fix 15: Verifikasi Akhir Kedua Dataset


In [90]:
print('=' * 55)
print('VERIFIKASI AKHIR — df_jobs (tech dataset)')
print('=' * 55)
print(f'Shape                  : {df_jobs.shape}')
print(f'Missing values total   : {df_jobs.isnull().sum().sum()}')
print(f'Duplikasi              : {df_jobs.duplicated().sum()}')
print(f'HTML tersisa           : {df_jobs["description"].str.contains("<[^>]+>", regex=True).sum()}')
print(f'Training-ready (flag)  : {df_jobs["is_train_ready"].sum()}')
print(f'Tech signal sources    :')
print(df_jobs['tech_signal_source'].value_counts().to_string())
print(f'Kolom                  : {df_jobs.columns.tolist()}')

print(f'\n{"=" * 55}')
print('VERIFIKASI AKHIR — df_resume')
print('=' * 55)
print(f'Shape            : {df_resume.shape}')
print(f'Missing values   : {df_resume.isnull().sum().sum()}')
print(f'Duplikasi        : {df_resume.duplicated().sum()}')
print(f'Kolom            : {df_resume.columns.tolist()}')

print(f'\n  Sample df_jobs bersih:')
df_jobs[['job_id','title','category','tech_signal_source','is_train_ready']].head(5)


VERIFIKASI AKHIR — df_jobs (tech dataset)
Shape                  : (2073, 35)
Missing values total   : 0
Duplikasi              : 0
HTML tersisa           : 0
Training-ready (flag)  : 2071
Tech signal sources    :
tech_signal_source
category|title           629
category                 552
title                    523
category|title|skills    272
title|skills              66
category|skills           31
Kolom                  : ['job_id', 'company_name', 'title', 'normalized_title', 'category', 'description', 'requirement_summary', 'work_type', 'employment_type', 'experience_level', 'province', 'city', 'salary_min', 'salary_max', 'salary_currency', 'salary_display', 'skills_top_10_names', 'requirements_concat', 'language_signal', 'source_posted_at', 'status', 'fit_input_quality_score', 'fit_input_has_requirements', 'fit_input_has_skills', 'skills_count_total', 'description_length_chars', 'req_len', '_is_tech_cat', '_is_tech_title', '_is_tech_skills', 'is_tech', 'tech_signal_source', 'h

,job_id,title,category,tech_signal_source,is_train_ready
2979,ffa33a78-6b46-4640-8002-1249292b2b5b,Frontend Web Developer,Frontend/Mobile Development,category|title|skills,1
1652,ff9f6a5c-69d3-4e93-bbc1-88472f8da8bb,DevSecOps,IT & Software,category,1
3073,ff9f2065-dce5-4531-bbf9-af45a779a939,Data Analyst/Data Scientist/Data Engineer/AI Engineer,Data & Analytics,category|title|skills,1
1036,ff60a17e-0a3d-4e10-883e-5592a95f0470,Senior Frontend Engineer,Technology,title,1
3518,ff53ad9e-438a-4926-8be3-2d36b9873092,Senior QA Automation Engineer,IT & Software,category|title|skills,1


In [91]:
print('  Sample df_resume bersih:')
df_resume.head(5)


  Sample df_resume bersih:


,ID,Skills,Projects,Education,Experience,Job_Role,Required_Skills
0,1,"HTML, JavaScript, Node.js, Manufacturing, SIEM, Electronics, Docker, Surgery","Healthcare App, AI Chatbot",S1,Fresher,Web Developer,"HTML, CSS, JavaScript, React, Node.js"
1,2,"APIs, Machine Learning, CSS, Visualization, PyTorch, Python, Network Security, CAD","Resume Analyzer, IoT Smart Home",D3,Fresher,ML Engineer,"Python, Deep Learning, TensorFlow, PyTorch, Machine Learning"
2,3,"React, Medical Knowledge, APIs, Python, Manufacturing, Git, Embedded Systems, RTOS","Stock Prediction Model, Cybersecurity Monitoring Tool",S2,3-5 years,Backend Developer,"Python, Java, SQL, APIs, Git"
3,4,"NumPy, Excel, Data Analysis, CSS, Python, Java, SolidWorks, Pandas","Portfolio Website, E-commerce Website",S1,1-2 years,Data Scientist,"Python, Machine Learning, Pandas, NumPy, Data Analysis"
4,5,"Kubernetes, Excel, Linux, Pharmacology, Manufacturing, C, Docker, RTOS","Stock Prediction Model, Portfolio Website",S1,3-5 years,Cloud Engineer,"AWS, Docker, Kubernetes, Linux, CI/CD"


### Simpan Dataset Bersih


In [97]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Dataset tech lengkap
jobs_eda_path    = processed_dir / 'indotech_job_cleaned.csv'
# Dataset training-ready saja (untuk AI model)
jobs_train_path  = processed_dir / 'indotech_job_train.csv'
# Dataset resume bersih
resume_path      = processed_dir / 'techtalent_profile_cleaned.csv'
# Dataset non-tech (untuk referensi, tidak dibuang)
nontech_path     = processed_dir / 'indotech_nontech.csv'

df_jobs.to_csv(jobs_eda_path, index=False, encoding='utf-8-sig')
df_jobs[df_jobs['is_train_ready'] == 1].to_csv(jobs_train_path, index=False, encoding='utf-8-sig')
df_resume.to_csv(resume_path, index=False, encoding='utf-8-sig')
df_nontech.to_csv(nontech_path, index=False, encoding='utf-8-sig')

n_train = df_jobs['is_train_ready'].sum()
print('Dataset bersih berhasil disimpan!')
print(f'\n  {processed_dir}/')
print(f'     ├── indotech_job_cleaned.csv       → {len(df_jobs):,} baris (EDA + Streamlit)')
print(f'     ├── indotech_job_train.csv         → {n_train:,} baris (AI model training)')
print(f'     ├── techtalent_profile_cleaned.csv → {len(df_resume):,} baris (resume)')
print(f'     └── indotech_nontech.csv           → {len(df_nontech):,} baris (non-tech, referensi)')

print(f'\nRingkasan Proses Cleaning:')
print(f'  df_jobs   : 2.099 → {len(df_jobs):,} tech jobs ({len(df_nontech):,} non-tech dipisah)')
print(f'  df_resume : 100.000 → {len(df_resume):,} baris ({100000-len(df_resume):,} baris dihapus)')
print(f'  Training-ready untuk model AI: {n_train:,} baris')



Dataset bersih berhasil disimpan!

  d:\BisaKerja\data\processed/
     ├── indotech_job_cleaned.csv       → 2,073 baris (EDA + Streamlit)
     ├── indotech_job_train.csv         → 2,071 baris (AI model training)
     ├── techtalent_profile_cleaned.csv → 69,929 baris (resume)
     └── indotech_nontech.csv           → 1,729 baris (non-tech, referensi)

Ringkasan Proses Cleaning:
  df_jobs   : 2.099 → 2,073 tech jobs (1,729 non-tech dipisah)
  df_resume : 100.000 → 69,929 baris (30,071 baris dihapus)
  Training-ready untuk model AI: 2,071 baris


## Exploratory Data Analysis (EDA)


## Visualization & Explanatory Analysis


In [96]:
# Pastikan df_jobs tersedia (jalankan setelah bagian Cleaning selesai)
print(f'df_jobs  : {df_jobs.shape[0]} baris x {df_jobs.shape[1]} kolom')
print(f'df_jobs: {df_jobs.shape[0]} baris x {df_jobs.shape[1]} kolom')
print(f'\nDistribusi kategori di df_jobs:')
print(df_jobs['category'].value_counts().to_string())
print(f'\nDistribusi experience_level:')
print(df_jobs['experience_level'].value_counts().to_string())


df_jobs  : 2073 baris x 35 kolom
df_jobs: 2073 baris x 35 kolom

Distribusi kategori di df_jobs:
category
IT & Software                                1212
Technology                                    168
Sains & Teknologi                              74
Information Technology & Communications        69
Operation & Technical Support                  56
Data & Analytics                               54
Product Management                             53
QA & Testing                                   49
Information Technology & Communication         46
Manufaktur, Transportasi & Logistik            38
Design & UX                                    25
Frontend/Mobile Development                    25
Design                                         22
Information Technology                         17
Business Analyst                               15
Desain & Arsitektur                             9
Science & Technology                            8
Sales & Marketing                           

### Pertanyaan 1: Apa 10 skill teknis yang paling banyak diminta oleh lowongan kerja tech di Indonesia berdasarkan data Mei 2026, dan berapa persentase kandidat dalam dataset resume yang sudah memiliki skill tersebut?
